# 1. Import Libraries

In [34]:
!pip install ollama

import pandas as pd
import ollama


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


# 2. Read Data

In [35]:
df = pd.read_csv("../dataset/dataset.csv")

# 3. Create Prompt Functions

In [ ]:
def clean_news_article(text, model="mistral:7b-instruct"):
    system_prompt = """
Anda adalah alat pembersih teks berita untuk NLP.

Hapus noise dari teks Bahasa Indonesia dan perbaiki kata yang rusak.

ATURAN:
- Hapus URL, email, metadata, dan label berita
- Perbaiki kata yang menempel akibat error (mis: dariABCnews -> dari ABCnews, trumpmembuat -> trump membuat)
- Jangan mengubah makna, urutan, atau isi teks

JANGAN PERNAH:
- menambahkan kalimat pembuka
- menambahkan judul
- menambahkan penjelasan apa pun
- menulis kalimat seperti “Berikut adalah teks…”
- menulis kalimat seperti “teks yang telah diolah…”
- menulis kalimat pengantar atau penutup
- menambahkan tanda kutip di awal atau akhir teks
- membungkus output dengan tanda kutip

OUTPUT:
Keluarkan hanya teks hasil pembersihan.
Tanpa tanda kutip, tanpa tambahan apa pun.
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ],
        options={
            "num_ctx": 6144,
            "temperature": 0
        }
    )

    return response["message"]["content"].strip()

def sentiment_analysis(text, model="mistral:7b-instruct"):
    system_prompt = """
Anda adalah mesin klasifikasi sentimen yang deterministik untuk artikel berita.

Tentukan sentimen keseluruhan teks sebagai SATU label saja:

Positive
Neutral
Negative

ATURAN LABEL:

Positive:
- Artikel terutama membahas keuntungan, peningkatan, keberhasilan, atau hasil yang positif.

Negative:
- Artikel terutama membahas kerugian, masalah, risiko, konflik, atau dampak negatif.

Neutral:
- Artikel bersifat faktual atau informatif tanpa dominasi jelas positif atau negatif.
- Termasuk laporan yang seimbang atau campuran.

ATURAN KETAT:
- Keluarkan HANYA satu kata: Positive, Neutral, atau Negative
- Tanpa penjelasan
- Tanpa tanda baca
- Tanpa teks tambahan
- Tanpa format apa pun
- Dasarkan keputusan hanya pada nada keseluruhan artikel
- Jika tidak yakin, pilih Neutral
"""

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": text}
        ],
        options={
            "num_ctx": 6144,
            "temperature": 0
        }
    )

    return response["message"]["content"].strip()

# 4. Run

In [ ]:
results = []
sentiments = []

for i, row in df.iterrows():
    try:
        cleaned = clean_news_article(row["konten"])
        sentiment = sentiment_analysis(cleaned)

        results.append(cleaned)
        sentiments.append(sentiment)

        print(i, cleaned, sentiment)
    except Exception as e:
        print(e)

df["cleaned"] = results
df["ollama_label"] = sentiments

# 5. Save to CSV

In [ ]:
df.to_csv("../outputs/dataset_ollama.csv")